# Cell-Akamai companion 81 — Workshop Module 9: Real agent + controlled orchestration

This extends upstream [`09_optional_agents_on_k8s.ipynb`](https://github.com/akamai-developers/akamai-workshop-ai-inference/blob/main/09_optional_agents_on_k8s/09_optional_agents_on_k8s.ipynb). It deploys the **actual upstream Solutions Architect agent** into your namespace if explicitly authorized, checks its responses and concurrency, samples vLLM live metrics, then executes B0–B3 scientific policy arms using the 32-item content-hash-frozen ECA dataset.

Important: ECA is synthetic; `B1_GENESIS` is a **proxy** and not the recovered Genesis model. No biological scientific inference, Anticube, ΔG*, or weight updates are claimed. All benchmarks are non-confirmatory pilots.

In [ ]:
import sys,os,json,time
from pathlib import Path
P=Path.cwd().resolve()
PACKAGE=next((p for p in (P,*P.parents) if (p/'cell_akamai_handoff.py').is_file()),None)
assert PACKAGE
sys.path.insert(0,str(PACKAGE))
import cell_akamai_handoff as cak
# Optional: explicitly point at the workshop git checkout if not auto-detected.
# os.environ['WORKSHOP_REPO']='/home/jovyan/akamai-workshop-ai-inference'
ENV=cak.preflight()
print('vLLM model:',ENV['served_model'],'namespace:',ENV['namespace'])
print('Frozen ECA input SHA256:',cak.ECA_SHA)
print('Workshop clone:',cak.locate_workshop())

## 1. Reuse upstream Module 9 deployment and tool code

The upstream agent has its own scope handler, a static `akamai_gpu_pricing` tool, Kubernetes ConfigMap, Deployment, Service and port-forward workflow. This cell **previews** those files. The next cell is the only deployment mutation gate. **It does not replace the upstream agent with a different stub.** Ensure Kubernetes NetworkPolicy permits agent → vLLM traffic.

In [ ]:
DEPLOY_PLAN=cak.deploy_upstream_agent(ENV,authorize=False)
print(json.dumps(DEPLOY_PLAN,indent=2))

## 2. Optional namespace-scoped agent deploy

**LIVE MUTATION:** Only set `AUTHORIZE_UPSTREAM_AGENT_DEPLOY=True` if you have permission to deploy this agent. The code uses the workshop's existing `agent.py`/`agent.yaml` and updates `MODEL_NAME` to the live served model. It will apply namespace resources and restart `sa-agent`, **not** vLLM. If you've already deployed the agent through upstream Module 9, leave `False`.

In [ ]:
AUTHORIZE_UPSTREAM_AGENT_DEPLOY=False # True only if authorized to deploy/restart the upstream agent
if AUTHORIZE_UPSTREAM_AGENT_DEPLOY:
    print(json.dumps(cak.deploy_upstream_agent(ENV,authorize=True),indent=2))
else:
    print('DEFERRED: upstream agent deployment not modified; existing sa-agent may still be tested.')

## 3. Execute the actual Module 9 agent: scope, pricing, and simultaneous requests

This uses `kubectl port-forward svc/sa-agent 18080:8080` and POSTs to the deployed agent. It tests in-scope/out-of-scope/model-disclosure/pricing questions and sends up to 6 simultaneous questions while sampling vLLM running/queued requests and KV cache. **Actual internal tool calls are not instrumented in upstream `agent.py`; a pricing answer alone is not verification that the tool was invoked.** Null metrics are not zeros. If no deployed service exists, this step fails without inventing measurements.

In [ ]:
RUN_UPSTREAM_AGENT=True
M09_AGENT=None
if RUN_UPSTREAM_AGENT:
    M09_AGENT=cak.exercise_upstream_agent(ENV,concurrency=6)
    print('Agent results:',M09_AGENT)
    print(json.dumps(cak.read_json(M09_AGENT/'agent_results.json'),indent=2)[:15000])
else:
    print('NOT_TESTED: upstream Kubernetes agent test deliberately skipped.')

## 4. Experimental B0–B3 orchestration on exactly the same served Akamai model

The content-hash-bound frozen ECA dataset is recovered from cell-akamai PR #7 (32 test inputs; rules 30/90/110/184). The runner **cannot access the oracle**. It sends 1 call for B0 and 3 calls for B1–B3 per item; this preserves Claude's existing policy prototype, but the B0 unequal-budget comparison and B2/B3 differing challenge informativeness remain important confounders. Set `N_ITEMS=8` for a bounded pilot or `N_ITEMS=32` for full pilot (320 model calls). Zero temperature; no self-training.

In [ ]:
N_ITEMS=8  # Optional 32 for full pilot: 320 model calls and potentially many minutes
M09_ECA=cak.run_eca(ENV,n_items=N_ITEMS,max_tokens=128,call_cap=3,token_cap=384)
print('Answer commitment:',cak.read_json(M09_ECA/'COMMIT.json'))

## 5. Independent ECA scoring — ONLY after answer commitment

The scorer refuses any altered `answers.jsonl`, `items.jsonl`, `config.json`, missing/duplicated arm×item rows, or mismatch with the frozen 32-item source. Exact oracle answers are calculated only in this later step. McNemar values for the pilot are descriptive, never confirmatory. No `ΔG*` or Anticube scalar is fabricated.

In [ ]:
M09_SCORE=cak.score_eca(M09_ECA)
print(json.dumps(M09_SCORE,indent=2))

## 6. What to show the judges

1. This notebook's `ENV` proving the actual model and dedicated vLLM endpoint; the Module 8 notebook's before/after table if an authorized rollout was executed.
2. Agent scope/tool-price output and 6-way concurrency with live KV/queue traces, showing the workshop's Module 9 agent on your inference.
3. A **separate** B0–B3 ECA accuracy/cost table, with held-out status `PILOT_NONCONFIRMATORY`, and explicitly note that B1 is a proxy.
4. Export using companion `82_verify_and_export.ipynb` (below). Return that ZIP here for independent re-analysis and dashboard ingestion.